# 05 — Fixed-Target Reliability Summary

Run independently: this notebook loads current completed DB models and champion
results itself. It exports **the fixed-target attainment PNG** plus native and
noise terminal-reliability CSVs. It does not compute adaptive AUC profiles, load
cross-function traces, run evaluations, or display images inline.

Detailed strategy/convergence profiles are in `06_convergence_ecdf_and_ablations.ipynb`;
noise robustness is in `07_noise_robustness.ipynb`. Each export is
cached separately and refreshed only when relevant inputs, settings or code change.


In [1]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "shared").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project directory or notebooks/analysis/.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

# Notebook-owned presentation helpers are imported from the project root.
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import plotly.io as pio
pio.defaults.mathjax = None
from benchmark.infra.di import BenchmarkProvider
from benchmark.application.benchmark_config import BenchmarkConfig, ReliabilityConfig
from benchmark.infra.storage.analysis_results_store import AnalysisResultsStore
from shared.infra.lifecycle import WorkflowScope
from benchmark.application.analysis.analyze_reliability import AnalyzeReliability
from shared.config import RESULTS_DIR
from notebooks.analysis.plotting.exporter import AnalysisFigureExporter
from notebooks.analysis.result_exports import AnalysisResultsExporter


figure_exporter = AnalysisFigureExporter(RESULTS_DIR)

# Optional condition filters; None selects the full available matrix.
FILTER_DIMS = None
FILTER_PROBLEMS = None
FILTER_NOISE_STDS = None


analysis_config = BenchmarkConfig(
    budget=100_000, trials_per_condition=20, random_seed=42, first_instance_id=2,
    trial_timeout_seconds=30.0, observation_trace_points=64,
    classical_baselines=["cmaes", "de", "pso"],
    noise_robustness_enabled=True, cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
    reliability=ReliabilityConfig(
        primary_prompt_strategy="baseline", primary_target=1e-8, secondary_target=1e-2,
        bootstrap_samples=1000, bootstrap_seed=20260923,
        checkpoint_fractions=[0.01, 0.1, 0.5, 1.0],
    ),
)

# Prepare services once; infrastructure owns shutdown cleanup.
if "services" in globals():
    services.close()
services = WorkflowScope(BenchmarkProvider(), context={}, request_context={BenchmarkConfig: analysis_config})
reliability_analysis = services.get(AnalyzeReliability)
results_store = services.get(AnalysisResultsStore)
results_exporter = AnalysisResultsExporter(results_store)


Diagnostic log: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/logs/benchmark/2026-10-30_14-31-06_UTC-eb5413db/diagnostics.log


## Calculate reliability (no figure or report export)


In [2]:
reliability_result = reliability_analysis.execute(dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
if reliability_result.is_success:
    reliability = reliability_result.value
    reliability_snapshot = results_exporter.save(reliability)
    print(f"Calculated {len(reliability.native)} current native condition rows.")
    display(reliability.native)
    
    print(f"Saved numerical results: {reliability_snapshot}")
else:
    reliability = None
    print(f"Error: {reliability_result.error.message}")


Calculated 1440 current native condition rows.


,Model,Strategy,Solver,Source Problem,Problem ID,Dim,Noise Std,Code Hash,Trials,Complete,Primary Successes,Primary Success Rate,Secondary Success Rate,CI Lower,CI Upper,Median Terminal Error
0,gwdg.qwen3-coder-next,baseline,gwdg.qwen3-coder-next_baseline,1,1,10,0.0,bf49989e38c0bee94920d3df969b34f6a0763b9e1455fa...,20,True,8,0.4,0.4,0.218807,0.613418,3.206428e+01
1,gwdg.qwen3-coder-next,baseline,gwdg.qwen3-coder-next_baseline_implicit,1,1,10,0.0,f62e10ebb9f4a363f98747b78b81f229ee5a789acfda8b...,20,True,18,0.9,0.9,0.698966,0.972134,2.842171e-14
2,gwdg.qwen3-coder-next,guided,gwdg.qwen3-coder-next_guided,1,1,10,0.0,a232d38ddcd03abeb5063fd7e010174cf1e7188475a44b...,20,True,20,1.0,1.0,0.838875,1.000000,3.552714e-15
3,gwdg.qwen3-coder-next,guided,gwdg.qwen3-coder-next_guided_implicit,1,1,10,0.0,6ec5170900f39157275cd21190fe539ae3948615f7a68b...,20,True,20,1.0,1.0,0.838875,1.000000,2.131628e-14
4,gwdg.qwen3-coder-next,thinking,gwdg.qwen3-coder-next_thinking,1,1,10,0.0,92b9ed8d77ec7dd37aa648cccd1084e567616ee9badfeb...,20,True,20,1.0,1.0,0.838875,1.000000,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,qwen2.5-coder-32b-instruct-q4_k_m.gguf,guided,qwen_32b_guided_noisy,8,8,50,0.2,bdc8738889060f4997384f98099929c0acf071d6e1f313...,20,True,0,0.0,0.0,0.000000,0.161125,4.523883e+02
1436,qwen2.5-coder-32b-instruct-q4_k_m.gguf,thinking,qwen_32b_thinking_implicit,8,8,50,0.2,ce866a0448912136d1c44ce75ec958f3a6abe2b30776fc...,20,True,0,0.0,0.0,0.000000,0.161125,2.774849e+05
1437,qwen2.5-coder-32b-instruct-q4_k_m.gguf,thinking,qwen_32b_thinking_noisy,8,8,50,0.2,c32bcf977bf9e3e976783fa1242ae720f1be4db60d8266...,20,True,0,0.0,0.0,0.000000,0.161125,6.671979e+02
1438,qwen2.5-coder-32b-instruct-q4_k_m.gguf,vectorization,qwen_32b_vectorization_implicit,8,8,50,0.2,9271028e8c2087cd78ff6422d051759f282a45adb83ae6...,20,True,0,0.0,0.0,0.000000,0.161125,2.386131e+05


Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/reliability/b7d93f2665f30fe64c6b68a25a65dd760fd04b7036a991180e63529a3f73c707


## Reliability tables and fixed-target attainment export

The matrix scores returned solutions. The attainment curve describes the best
clean point queried, which need not be returned. Noise reliability uses identical
frozen champions, not separately synthesized noisy champions. Incomplete conditions
are excluded from aggregate figures, not counted as failures.


In [3]:
if reliability is not None:
    figure_exporter.export_reliability(reliability)
else:
    print("Complete the preceding workflow successfully before using its data.")


ExportSummary(written=(PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/06_thesis_summary/fig_10b_fixed_target_attainment.png'),), skipped=())